In [1]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
import os 

In [6]:
# ── Step 1: Define the tool ─────────────────────────────────────
@tool
def add_numbers(numbers: list[int]) -> int:
    """Adds a list of numbers together. Use this when asked to add or sum numbers."""
    return sum(numbers)

In [7]:
# ── Step 2: Set up Groq LLM and bind the tool ───────────────────
# Initialize the Groq LLM
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-120b",  # change to your target model
    temperature=0.7,
    max_tokens=1024,
)


llm_with_tools = llm.bind_tools([add_numbers])

In [14]:
# ── Step 3: The function that handles everything ─────────────────
def chat(user_question: str):
    print(f"\n User: {user_question}")
    print("-" * 50)

    messages = [HumanMessage(user_question)]

    # LLM decides: do I need a tool or can I answer directly?
    response = llm_with_tools.invoke(messages)

    # ── Case A: LLM decided NO tool needed ──────────────────────
    if not response.tool_calls:
        print(f" LLM answered directly (no tool called)")
        print(f" Answer: {response.content}")
        return

    # ── Case B: LLM decided YES, call a tool ────────────────────
    print(f" LLM decided to call tool: '{response.tool_calls[0]['name']}'")
    print(f" With args: {response.tool_calls[0]['args']}")

    # Execute the tool
    tool_result = add_numbers.invoke(response.tool_calls[0]["args"])
    print(f" Tool returned: {tool_result}")

    # Feed result back to LLM for a natural language answer
    messages += [
        response,  # LLM's decision to call tool
        ToolMessage(  # tool's result
            content=str(tool_result),
            tool_call_id=response.tool_calls[0]["id"]
        )
    ]

    final = llm_with_tools.invoke(messages)
    print(f" Final Answer: {final.content}")


# ── Test both cases ──────────────────────────────────────────────
chat("Add these numbers 1, 2, 3, 54")
chat("What is the capital of India?")



 User: Add these numbers 1, 2, 3, 54
--------------------------------------------------
 LLM decided to call tool: 'add_numbers'
 With args: {'numbers': [1, 2, 3, 54]}
 Tool returned: 60
 Final Answer: The sum of 1 + 2 + 3 + 54 is **60**.

 User: What is the capital of India?
--------------------------------------------------
 LLM answered directly (no tool called)
 Answer: The capital of India is **New Delhi**.
